# Load Dataset

In [87]:
import pandas as pd

In [88]:
gk_df = pd.read_csv('/content/gk_qna_dataset.csv')

gk_df

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo
...,...,...
166,"Can you tell me, ""Which bird cannot fly""?",Ostrich
167,"I was wondering, ""What is 'H2O' commonly called""?",Water
168,"I was wondering, ""Which planet is known as the...",Mars
169,"Can you tell me, ""What is the main religion in...",Islam


In [89]:
gk_df['question']

,question
0,"What is the capital of ""France""?"
1,"What is the capital of ""Germany""?"
2,"What is the capital of ""Italy""?"
3,"What is the capital of ""Spain""?"
4,"What is the capital of ""Japan""?"
...,...
166,"Can you tell me, ""Which bird cannot fly""?"
167,"I was wondering, ""What is 'H2O' commonly called""?"
168,"I was wondering, ""Which planet is known as the..."
169,"Can you tell me, ""What is the main religion in..."


In [90]:
gk_df['question'][0]

'What is the capital of "France"?'

# Tokenization


In [91]:
import re

def tokenize(text):

  #1. LowerCase
  text = text.lower()

  #2. Remove Quotes
  text = re.sub(r"[\"']","",text)

  #3. Remove all punctuations
  text = re.sub(r"[^a-z0-9\$]"," ", text)

  #4. Remove extra spaces
  text = re.sub(r"\s+"," ",text).strip()


  #Tokenize split into words
  tokens = text.split()

  return tokens

In [92]:
print(tokenize(gk_df['question'][0]))

['what', 'is', 'the', 'capital', 'of', 'france']


# Vocab Forming


In [93]:
vocab = {"<UNKNOWN>":0}

In [94]:
def build_vocab(row):
  # Tokenize question and answer
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  # Combine tokens from question and answer
  merged_tokens = tokenized_question + tokenized_answer

  # Add new tokens to the vocabulary
  for token in merged_tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

In [95]:
gk_df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [96]:
vocab

{'<UNKNOWN>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'italy': 10,
 'rome': 11,
 'spain': 12,
 'madrid': 13,
 'japan': 14,
 'tokyo': 15,
 'canada': 16,
 'ottawa': 17,
 'brazil': 18,
 'brasilia': 19,
 'australia': 20,
 'canberra': 21,
 'india': 22,
 'new': 23,
 'delhi': 24,
 'china': 25,
 'beijing': 26,
 'russia': 27,
 'moscow': 28,
 'united': 29,
 'states': 30,
 'washington': 31,
 'dc': 32,
 'mexico': 33,
 'city': 34,
 'egypt': 35,
 'cairo': 36,
 'turkey': 37,
 'ankara': 38,
 'argentina': 39,
 'buenos': 40,
 'aires': 41,
 'south': 42,
 'korea': 43,
 'seoul': 44,
 'indonesia': 45,
 'jakarta': 46,
 'pakistan': 47,
 'islamabad': 48,
 'bangladesh': 49,
 'dhaka': 50,
 'nepal': 51,
 'kathmandu': 52,
 'sri': 53,
 'lanka': 54,
 'colombo': 55,
 'thailand': 56,
 'bangkok': 57,
 'malaysia': 58,
 'kuala': 59,
 'lumpur': 60,
 'vietnam': 61,
 'hanoi': 62,
 'uae': 63,
 'abu': 64,
 'dhabi': 65,
 'iran': 66,
 'tehran': 67,
 '

In [97]:
len(vocab)

243

In [98]:
def text_to_indecies(text,vocab):
  index_text = []

  for token in tokenize(text):

    if token in vocab:
      index_text.append(vocab[token])
    else:
      index_text.append(vocab['<UNKNOWN>'])

  return index_text


In [99]:
text_to_indecies('the capital of france',vocab)

[3, 4, 5, 6]

# Dataset and Dataloader

In [100]:
import torch
from torch.utils.data import Dataset,DataLoader

In [101]:
gk_df.shape[0]

171

In [102]:
index=0

text_to_indecies(gk_df.iloc[index]['answer'],vocab)

[7]

In [103]:
class QADataset(Dataset):

  def __init__(self,gk_df,vocab):

    self.gk_df = gk_df
    self.vocab = vocab

  def __len__(self):

    return self.gk_df.shape[0]

  def __getitem__(self, index):

    numerical_question = text_to_indecies(self.gk_df.iloc[index]['question'],self.vocab)
    numerical_answer= text_to_indecies(self.gk_df.iloc[index]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)



In [104]:
dataset = QADataset(gk_df,vocab)


In [105]:
dataset[1]

(tensor([1, 2, 3, 4, 5, 8]), tensor([9]))

In [106]:
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [107]:
for question,answer in dataloader:
  print(question,answer[0])

tensor([[ 1,  2,  3,  4,  5, 14]]) tensor([15])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([9])
tensor([[ 1,  2,  3,  4,  5, 53, 54]]) tensor([55])
tensor([[ 93, 191, 192, 193]]) tensor([194])
tensor([[237,   1, 238, 239,  79, 170, 174, 175]]) tensor([176, 177, 178])
tensor([[ 93, 215, 216, 213]]) tensor([217])
tensor([[ 1,  2,  3,  4,  5, 70, 71]]) tensor([72])
tensor([[ 95, 232, 233,   1,   2,   3, 111, 112]]) tensor([113, 112])
tensor([[ 93, 191, 192, 193,  95, 232, 233]]) tensor([194])
tensor([[  1,   2,   3, 220, 129,   5,  18]]) tensor([221])
tensor([[ 1,  2,  3,  4,  5, 47]]) tensor([48])
tensor([[234, 235, 236,  93, 202, 203, 204]]) tensor([205])
tensor([[ 79, 105,   3, 106, 107,  95, 232, 233]]) tensor([108, 109, 110])
tensor([[  1,   2, 149, 150, 121]]) tensor([151, 152])
tensor([[234, 235, 236,  79,  84,   3, 161, 162]]) tensor([163, 164])
tensor([[237,   1, 238, 239,   1,   2, 141, 142]]) tensor([143, 144])
tensor([[240, 232, 241, 242,   1,   2, 141, 142]]) tensor([143, 144])
tens

In [108]:
# #Squeeze
# def squeeze_fun():


In [109]:
import torch.nn as nn

class simpleNN(nn.Module):

  def __init__(self, vocab_size, embedding_dim=50, hidden_size=64):
    super().__init__()

    # Converts word indices to dense vectors
    self.embedding = nn.Embedding(vocab_size, embedding_dim)

    # Processes sequence data. batch_first=True means we use (Batch, Seq, Feature) format
    self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    # Maps the RNN output back to the vocabulary size for prediction
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, question):

    embedded = self.embedding(question) # (1,seq_len,50)
    # embedded shape: (batch_size, sequence_length, embedding_dim)

    _, final = self.rnn(embedded)
    # output shape: (batch_size, sequence_length, hidden_size)

    return self.fc(final.squeeze(0)) # (1,vocab_size)


In [110]:
model = simpleNN(vocab_size=len(vocab))

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

epochs = 50


In [111]:
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
      output = model(question)   # (1,vocab_size)

      target = answer[0][0].unsqueeze(0)   #take only the first answer token as target -> shape (1,)

      loss = loss_fn(output,target)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      total_loss += loss.item()

    if (epoch + 1) % 10 ==0:
      print(f"Epoch{epoch+1:3d}/{epochs} | Loss: {total_loss:.4f}")

Epoch 10/50 | Loss: 607.6559
Epoch 20/50 | Loss: 380.2482
Epoch 30/50 | Loss: 253.3389
Epoch 40/50 | Loss: 176.2965
Epoch 50/50 | Loss: 122.6973


In [112]:
vocab.items()

dict_items([('<UNKNOWN>', 0), ('what', 1), ('is', 2), ('the', 3), ('capital', 4), ('of', 5), ('france', 6), ('paris', 7), ('germany', 8), ('berlin', 9), ('italy', 10), ('rome', 11), ('spain', 12), ('madrid', 13), ('japan', 14), ('tokyo', 15), ('canada', 16), ('ottawa', 17), ('brazil', 18), ('brasilia', 19), ('australia', 20), ('canberra', 21), ('india', 22), ('new', 23), ('delhi', 24), ('china', 25), ('beijing', 26), ('russia', 27), ('moscow', 28), ('united', 29), ('states', 30), ('washington', 31), ('dc', 32), ('mexico', 33), ('city', 34), ('egypt', 35), ('cairo', 36), ('turkey', 37), ('ankara', 38), ('argentina', 39), ('buenos', 40), ('aires', 41), ('south', 42), ('korea', 43), ('seoul', 44), ('indonesia', 45), ('jakarta', 46), ('pakistan', 47), ('islamabad', 48), ('bangladesh', 49), ('dhaka', 50), ('nepal', 51), ('kathmandu', 52), ('sri', 53), ('lanka', 54), ('colombo', 55), ('thailand', 56), ('bangkok', 57), ('malaysia', 58), ('kuala', 59), ('lumpur', 60), ('vietnam', 61), ('hanoi'

In [113]:
idx_to_word= {idx : word for word,idx in vocab.items()}
idx_to_word

{0: '<UNKNOWN>',
 1: 'what',
 2: 'is',
 3: 'the',
 4: 'capital',
 5: 'of',
 6: 'france',
 7: 'paris',
 8: 'germany',
 9: 'berlin',
 10: 'italy',
 11: 'rome',
 12: 'spain',
 13: 'madrid',
 14: 'japan',
 15: 'tokyo',
 16: 'canada',
 17: 'ottawa',
 18: 'brazil',
 19: 'brasilia',
 20: 'australia',
 21: 'canberra',
 22: 'india',
 23: 'new',
 24: 'delhi',
 25: 'china',
 26: 'beijing',
 27: 'russia',
 28: 'moscow',
 29: 'united',
 30: 'states',
 31: 'washington',
 32: 'dc',
 33: 'mexico',
 34: 'city',
 35: 'egypt',
 36: 'cairo',
 37: 'turkey',
 38: 'ankara',
 39: 'argentina',
 40: 'buenos',
 41: 'aires',
 42: 'south',
 43: 'korea',
 44: 'seoul',
 45: 'indonesia',
 46: 'jakarta',
 47: 'pakistan',
 48: 'islamabad',
 49: 'bangladesh',
 50: 'dhaka',
 51: 'nepal',
 52: 'kathmandu',
 53: 'sri',
 54: 'lanka',
 55: 'colombo',
 56: 'thailand',
 57: 'bangkok',
 58: 'malaysia',
 59: 'kuala',
 60: 'lumpur',
 61: 'vietnam',
 62: 'hanoi',
 63: 'uae',
 64: 'abu',
 65: 'dhabi',
 66: 'iran',
 67: 'tehran',
 6

In [114]:
# Example: How to use your new dictionary to 'read' the model's mind
sample_index = 7
word = idx_to_word[sample_index]

print(f"If the model predicts index {sample_index}, the word is: '{word}'")

If the model predicts index 7, the word is: 'paris'


In [115]:
def predict(question_text,model,vocab,idx_to_word):
  model.eval()

  with torch.no_grad():
    indices = text_to_indecies(question_text,vocab)

    if not indices:
      return "<UNKNOWN>"

    x = torch.tensor(indices).unsqueeze(0)

    pred = torch.argmax(model(x),dim=1).item()

  return idx_to_word.get(pred,"<UNKNOWN>")

In [116]:
tests = [
    "What is the capital of France?",
    "What is H2O commonly called?",
    "Which bird cannot fly?",
    "What is the capital of Japan?",
    "Which planet is known as the red planet?"
]

for q in tests:
    # Fixed the f-string syntax here
    prediction = predict(q, model, vocab, idx_to_word)
    print(f"{q:45} -> {prediction}")

What is the capital of France?                -> paris
What is H2O commonly called?                  -> water
Which bird cannot fly?                        -> ostrich
What is the capital of Japan?                 -> tokyo
Which planet is known as the red planet?      -> mars
